# Drone Tracker — Colab Setup

Notebook do uruchomienia projektu drone-tracker-system w Google Colab.

**Wymagania**: Colab (free wystarczy dla start), GPU runtime włączone.

**Kolejność użycia**:
1. Runtime → Change runtime type → T4 GPU
2. Uruchamiaj komórki po kolei
3. Mount Google Drive z Twoim datasetem
4. Eval baseline / fine-tune / inference


## Krok 1 — Weryfikacja GPU

In [ ]:
!nvidia-smi

Powinno pokazać T4 GPU (lub V100/A100 jeśli masz Pro). Jeśli nie:
- Runtime → Change runtime type → T4 GPU → Save
- Uruchom komórkę ponownie

## Krok 2 — Instalacja zależności

Colab ma już numpy, opencv, pytorch. Dodajemy ultralytics + weryfikujemy.

In [ ]:
!pip install -q ultralytics==8.4.30

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

from ultralytics import YOLO
print(f'Ultralytics OK')

## Krok 3 — Mount Google Drive

Trzymamy dataset i wyniki na Drive — przetrwają reset sesji.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Utwórz strukturę folderów dla projektu na Drive
import os
PROJECT_DIR = '/content/drive/MyDrive/drone_tracker'
os.makedirs(f'{PROJECT_DIR}/datasets', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/runs', exist_ok=True)
print(f'Struktura utworzona: {PROJECT_DIR}')
!ls -la {PROJECT_DIR}

## Krok 4 — Baseline eval YOLOv8s na drone images

Szybki sanity check. Wrzuć kilka drone images do `{PROJECT_DIR}/datasets/drone_test/`.
Możesz je pobrać ręcznie z Pexels albo z jakiegoś public dataset.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # pobierze automatycznie ~22 MB

# Test na pojedynczym obrazie — podmień ścieżkę
TEST_IMAGE = f'{PROJECT_DIR}/datasets/drone_test/example.jpg'

if os.path.exists(TEST_IMAGE):
    results = model.predict(TEST_IMAGE, conf=0.05, imgsz=960, save=True)
    for r in results:
        print(f'Detekcje: {len(r.boxes) if r.boxes is not None else 0}')
        if r.boxes is not None:
            for i in range(len(r.boxes)):
                cls = int(r.boxes.cls[i].item())
                conf = float(r.boxes.conf[i].item())
                print(f'  class={cls} ({r.names.get(cls)}) conf={conf:.2f}')
    print(f'\nZapisano wynik z ramkami do: runs/detect/predict/')
else:
    print(f'Brak pliku {TEST_IMAGE}. Wrzuć drone image do folderu drone_test/ na Drive.')

## Krok 5 — Pobranie publicznego dataset

**Det-Fly** — DJI drone images, public GitHub, direct download bez auth.
~13k obrazów, ~2 GB.

In [ ]:
# Det-Fly — wymaga clone od GitHub
# Alternatywa: Roboflow Universe z API (wymaga konta)

DATASET_DIR = f'{PROJECT_DIR}/datasets/det_fly'

if not os.path.exists(DATASET_DIR):
    !git clone https://github.com/Jake-WU/Det-Fly.git {DATASET_DIR}
    print(f'Pobrano Det-Fly do {DATASET_DIR}')
else:
    print(f'Det-Fly już jest w {DATASET_DIR}')

!ls {DATASET_DIR} | head -20

## Krok 6 — Konwersja annotacji do YOLO format

Każdy dataset ma inny format annotacji (XML PASCAL VOC, JSON COCO, custom).
Ultralytics oczekuje YOLO format: `class x_center y_center width height` (znormalizowane 0-1).

**Template konwersji** poniżej. Dostosuj do konkretnego datasetu.

In [ ]:
# Template: PASCAL VOC XML -> YOLO txt
# Skopiuj i adaptuj dla swojego datasetu

import xml.etree.ElementTree as ET
from pathlib import Path

def voc_to_yolo(xml_path, output_txt, img_w, img_h, class_map={'drone': 0}):
    tree = ET.parse(xml_path)
    lines = []
    for obj in tree.findall('object'):
        cls_name = obj.find('name').text
        if cls_name not in class_map:
            continue
        cls_id = class_map[cls_name]
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)
        xc = (xmin + xmax) / 2.0 / img_w
        yc = (ymin + ymax) / 2.0 / img_h
        w = (xmax - xmin) / img_w
        h = (ymax - ymin) / img_h
        lines.append(f'{cls_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}')
    Path(output_txt).write_text('\n'.join(lines))

# Użycie — adaptuj ścieżki:
# voc_to_yolo('annotations/img001.xml', 'labels/img001.txt', 1920, 1080)

## Krok 7 — Training yolov8m na drone dataset

Szablon komendy. Dataset musi być w YOLO format z `data.yaml`:

```yaml
path: /content/drive/MyDrive/drone_tracker/datasets/drone_v1
train: images/train
val: images/val
test: images/test
names:
  0: drone
```

In [ ]:
# Basic training — zacznij od tego
from ultralytics import YOLO

# Załaduj pretrained yolov8m
model = YOLO('yolov8m.pt')

# Train
results = model.train(
    data=f'{PROJECT_DIR}/datasets/drone_v1/data.yaml',
    epochs=100,
    imgsz=1280,
    batch=8,          # T4 16GB — zmniejsz do 4 jeśli OOM
    device=0,
    name='drone_v1',
    project=f'{PROJECT_DIR}/runs',
    patience=20,
    save=True,
    plots=True,
)

## Krok 8 — Training yolov8m z small-object augmentations

Dla dronów 2-30 px w kadrze. Agresywne augmentacje.

In [ ]:
model = YOLO('yolov8m.pt')

results = model.train(
    data=f'{PROJECT_DIR}/datasets/drone_v1/data.yaml',
    epochs=150,
    imgsz=1920,       # wyższa rozdzielczość dla małych obiektów
    batch=4,          # batch=4 dla imgsz=1920 na T4
    device=0,
    name='drone_small_v1',
    project=f'{PROJECT_DIR}/runs',
    patience=25,
    # Augmentations dla małych obiektów:
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.3,
    scale=0.5,
    close_mosaic=10,
    save=True,
    plots=True,
)

## Krok 9 — Eval custom model na test set

In [ ]:
# Załaduj najlepsze weights z trainingu
model = YOLO(f'{PROJECT_DIR}/runs/drone_small_v1/weights/best.pt')

# Eval na test set
metrics = model.val(
    data=f'{PROJECT_DIR}/datasets/drone_v1/data.yaml',
    split='test',
    imgsz=1920,
    conf=0.25,
)

print(f'mAP@0.5:     {metrics.box.map50:.4f}')
print(f'mAP@0.5-0.95: {metrics.box.map:.4f}')
print(f'Precision:   {metrics.box.mp:.4f}')
print(f'Recall:      {metrics.box.mr:.4f}')

## Krok 10 — Inference na własnym video

Wrzuć `video.mp4` na Drive, uruchom custom model, zobacz wynik.

In [ ]:
# Podmień ścieżkę do video
VIDEO_PATH = f'{PROJECT_DIR}/video.mp4'

model = YOLO(f'{PROJECT_DIR}/runs/drone_small_v1/weights/best.pt')

results = model.predict(
    source=VIDEO_PATH,
    conf=0.25,
    imgsz=1920,
    save=True,
    project=f'{PROJECT_DIR}/runs/inference',
    name='test_video',
)

print(f'Wynik zapisany w: {PROJECT_DIR}/runs/inference/test_video/')
print(f'Video z ramkami: .mp4 w tym folderze')

## Krok 11 — Download wyników

Gdy training skończony, weights będą na Drive. Możesz je pobrać lokalnie:

```python
from google.colab import files
files.download(f'{PROJECT_DIR}/runs/drone_small_v1/weights/best.pt')
```

Albo skopiować ręcznie z Drive na własny dysk (~20-100 MB).

---

**Po skończeniu trainingu** — skopiuj `best.pt` do lokalnego repo jako
`yolov8_drone.pt`, zmień `config/config.yaml`:

```yaml
yolo:
  model: yolov8_drone.pt
  classes: [0]  # drone = klasa 0 w custom modelu
```

Uruchom pipeline normalnie — teraz będzie używał Twojego fine-tuned modelu.